# Modelagem Preditiva e Segmentação de Clientes

Este notebook implementa dois pilares de modelagem:
1. **Previsão de Satisfação do Cliente** — Classificação binária (Logistic Regression + Random Forest)
2. **Segmentação de Clientes (RFM)** — Clusterização K-Means sobre métricas de Recência, Frequência e Monetário

In [ ]:
# =============================================================================
# Célula 1: Importações e Configuração
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, accuracy_score, f1_score
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Configurações visuais
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Diretório para salvar figuras
os.makedirs('../reports/figures', exist_ok=True)

print('Importações concluídas.')

In [ ]:
# =============================================================================
# Célula 2: Carga e Preparação dos Dados
# =============================================================================
dados = pd.read_csv('../dados_integrados_preprocessados.csv')

# Converter timestamps
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    dados[col] = pd.to_datetime(dados[col], errors='coerce')

print(f'Dataset carregado: {dados.shape[0]} linhas, {dados.shape[1]} colunas')
print(f'Colunas: {dados.columns.tolist()}')

---
## Parte 1: Previsão de Satisfação do Cliente (Classificação)

**Objetivo:** Prever se um pedido terá avaliação **positiva** (review_score >= 4) ou **negativa** (review_score < 4).

**Modelos:** Logistic Regression e Random Forest Classifier.

In [ ]:
# =============================================================================
# Célula 3: Feature Engineering para Classificação
# =============================================================================

# Filtrar apenas pedidos entregues com review
df_model = dados[
    (dados['order_status'] == 'delivered') &
    (dados['review_score'].notna()) &
    (dados['order_delivered_customer_date'].notna()) &
    (dados['order_estimated_delivery_date'].notna())
].copy()

# --- Engenharia de Features ---
# Dias de entrega (data entregue - data compra)
df_model['delivery_days'] = (
    df_model['order_delivered_customer_date'] - df_model['order_purchase_timestamp']
).dt.total_seconds() / 86400

# Atraso na entrega (dias entregue - dias estimados). Positivo = atrasado
df_model['delivery_delay'] = (
    df_model['order_delivered_customer_date'] - df_model['order_estimated_delivery_date']
).dt.total_seconds() / 86400

# Flag: entregou com atraso?
df_model['is_late'] = (df_model['delivery_delay'] > 0).astype(int)

# Diferença entre preço e frete (proporção frete/preço)
df_model['freight_ratio'] = df_model['freight_value'] / (df_model['price'] + 1)

# Variável alvo: satisfeito (1) vs insatisfeito (0)
df_model['satisfeito'] = (df_model['review_score'] >= 4).astype(int)

print(f'Registros para modelagem: {len(df_model)}')
print(f'\nDistribuição da variável alvo:')
print(df_model['satisfeito'].value_counts(normalize=True).round(3))
print(f'\nNovas features: delivery_days, delivery_delay, is_late, freight_ratio')

In [ ]:
# =============================================================================
# Célula 4: Preparação dos Dados para Treino
# =============================================================================

# Features numéricas selecionadas
feature_cols = [
    'price', 'freight_value', 'payment_value', 'payment_installments',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'delivery_days', 'delivery_delay', 'is_late', 'freight_ratio'
]

# Remover linhas com NaN nas features
df_clean = df_model[feature_cols + ['satisfeito']].dropna()

# Amostra para performance (max 20K registros)
MAX_SAMPLES = 20000
if len(df_clean) > MAX_SAMPLES:
    df_clean = df_clean.sample(MAX_SAMPLES, random_state=42)
    print(f'Amostra reduzida a {MAX_SAMPLES} registros para performance')

X = df_clean[feature_cols]
y = df_clean['satisfeito']

# Dividir treino/teste (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Normalizar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')
print(f'Features: {feature_cols}')

In [ ]:
# =============================================================================
# Célula 5: Modelo 1 — Logistic Regression
# =============================================================================

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Previsões
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Métricas
print('=' * 60)
print('LOGISTIC REGRESSION — Resultados')
print('=' * 60)
print(f'Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_lr):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Insatisfeito', 'Satisfeito']))

In [ ]:
# =============================================================================
# Célula 6: Modelo 2 — Random Forest Classifier
# =============================================================================

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

# Previsões
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Métricas
print('=' * 60)
print('RANDOM FOREST CLASSIFIER — Resultados')
print('=' * 60)
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Insatisfeito', 'Satisfeito']))

In [ ]:
# =============================================================================
# Célula 7: Comparação Visual — Curvas ROC
# =============================================================================

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})', linewidth=2)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curva ROC — Comparação de Modelos de Satisfação')
ax.legend(loc='lower right', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/modelo_1_curva_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: reports/figures/modelo_1_curva_roc.png')

In [ ]:
# =============================================================================
# Célula 8: Matrizes de Confusão (lado a lado)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Insatisfeito', 'Satisfeito'],
            yticklabels=['Insatisfeito', 'Satisfeito'])
axes[0].set_title('Logistic Regression')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Previsto')

# Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Insatisfeito', 'Satisfeito'],
            yticklabels=['Insatisfeito', 'Satisfeito'])
axes[1].set_title('Random Forest')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Previsto')

plt.suptitle('Matrizes de Confusão — Previsão de Satisfação', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/modelo_2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: reports/figures/modelo_2_confusion_matrix.png')

In [ ]:
# =============================================================================
# Célula 9: Feature Importance (Random Forest)
# =============================================================================

importances = pd.Series(
    rf_model.feature_importances_, index=feature_cols
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(importances)))
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Importância das Features — Random Forest (Satisfação)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importância')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/modelo_3_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 features mais importantes:')
for feat, imp in importances.tail(5).items():
    print(f'  {feat}: {imp:.4f}')
print('\nFigura salva: reports/figures/modelo_3_feature_importance.png')

In [ ]:
# =============================================================================
# Célula 10: Cross-Validation (5-Fold) para Robustez
# =============================================================================

print('Cross-Validation (5-Fold) — ROC-AUC:')
print('-' * 40)

# Logistic Regression
cv_lr = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f'Logistic Regression:  {cv_lr.mean():.4f} +/- {cv_lr.std():.4f}')

# Random Forest
cv_rf = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f'Random Forest:        {cv_rf.mean():.4f} +/- {cv_rf.std():.4f}')

# Tabela Resumo
resumo = pd.DataFrame({
    'Modelo': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_rf)],
    'F1-Score': [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_rf)],
    'ROC-AUC (test)': [roc_auc_score(y_test, y_prob_lr), roc_auc_score(y_test, y_prob_rf)],
    'ROC-AUC CV (mean)': [cv_lr.mean(), cv_rf.mean()],
    'ROC-AUC CV (std)': [cv_lr.std(), cv_rf.std()]
}).round(4)

print('\n--- Tabela Comparativa ---')
print(resumo.to_string(index=False))

# Salvar tabela
resumo.to_csv('../reports/tabela_comparativa_modelos.csv', index=False)
print('\nTabela salva: reports/tabela_comparativa_modelos.csv')

---
## Parte 2: Segmentação de Clientes (RFM + K-Means)

**Objetivo:** Segmentar clientes usando métricas de **Recência**, **Frequência** e valor **Monetário** (RFM) para identificar perfis de consumo.

**Técnica:** K-Means Clustering com análise do número óptimo de clusters (Elbow + Silhouette).

In [ ]:
# =============================================================================
# Célula 11: Cálculo RFM (Recência, Frequência, Monetário)
# =============================================================================

# Filtrar pedidos entregues
df_rfm = dados[
    (dados['order_status'] == 'delivered') &
    (dados['order_purchase_timestamp'].notna()) &
    (dados['customer_unique_id'].notna()) &
    (dados['payment_value'].notna())
].copy()

# Data de referência (dia após a última compra)
ref_date = df_rfm['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f'Data de referência: {ref_date.date()}')

# Calcular RFM por cliente
rfm = df_rfm.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (ref_date - x.max()).days,  # Recência (dias)
    'order_id': 'nunique',                                             # Frequência
    'payment_value': 'sum'                                             # Monetário
}).rename(columns={
    'order_purchase_timestamp': 'Recencia',
    'order_id': 'Frequencia',
    'payment_value': 'Monetario'
})

print(f'\nClientes únicos: {len(rfm)}')
print('\nEstatísticas RFM:')
print(rfm.describe().round(2))

In [ ]:
# =============================================================================
# Célula 12: Distribuição RFM
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(['Recencia', 'Frequencia', 'Monetario']):
    sns.histplot(rfm[col], bins=50, kde=True, ax=axes[i], color=['#3498db', '#2ecc71', '#e74c3c'][i])
    axes[i].set_title(f'Distribuição: {col}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)

plt.suptitle('Distribuição das Métricas RFM', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/modelo_4_rfm_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: reports/figures/modelo_4_rfm_distribuicao.png')

In [ ]:
# =============================================================================
# Célula 13: Método do Cotovelo + Silhouette para K Ótimo
# =============================================================================

# Normalizar RFM
scaler_rfm = StandardScaler()
rfm_scaled = scaler_rfm.fit_transform(rfm)

# Amostra para Silhouette (operação O(n²), limitar a 15K)
SAMPLE_SIZE = min(15000, len(rfm_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(rfm_scaled), SAMPLE_SIZE, replace=False)
rfm_sample = rfm_scaled[sample_idx]
print(f'Usando amostra de {SAMPLE_SIZE} clientes para Elbow/Silhouette')

# Testar K de 2 a 8
K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_sample)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_sample, labels))
    print(f'K={k}: Inércia={km.inertia_:.0f}, Silhouette={silhouettes[-1]:.4f}')

# Gráficos
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_title('Método do Cotovelo (Elbow)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 'rs-', linewidth=2)
axes[1].set_title('Silhouette Score', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Score')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Determinação do K Ótimo', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/modelo_5_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: reports/figures/modelo_5_elbow_silhouette.png')

In [ ]:
# =============================================================================
# Célula 14: Clusterização Final (K=4) — Aplicado ao dataset completo
# =============================================================================

K_FINAL = 4

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
rfm['Cluster'] = kmeans_final.fit_predict(rfm_scaled)

# Perfil de cada cluster
perfil = rfm.groupby('Cluster').agg({
    'Recencia': 'mean',
    'Frequencia': 'mean',
    'Monetario': 'mean'
}).round(2)

# Contar clientes por cluster
perfil['N_Clientes'] = rfm.groupby('Cluster').size()
perfil['Pct_Clientes'] = (perfil['N_Clientes'] / len(rfm) * 100).round(1)

print('Perfil dos Clusters RFM:')
print('=' * 70)
print(perfil)
# Silhouette com amostra para performance
sil_idx = np.random.choice(len(rfm_scaled), min(15000, len(rfm_scaled)), replace=False)
sil_score = silhouette_score(rfm_scaled[sil_idx], rfm['Cluster'].values[sil_idx])
print(f'\nSilhouette Score (K={K_FINAL}, amostra): {sil_score:.4f}')

In [ ]:
# =============================================================================
# Célula 15: Nomeação Automática dos Clusters
# =============================================================================

# Determinar nomes com base nas características RFM
def nomear_cluster(row):
    rec_med = rfm['Recencia'].median()
    freq_med = rfm['Frequencia'].median()
    mon_med = rfm['Monetario'].median()
    
    if row['Recencia'] <= rec_med and row['Monetario'] >= mon_med:
        return 'VIP'
    elif row['Recencia'] <= rec_med and row['Monetario'] < mon_med:
        return 'Novos / Promissores'
    elif row['Recencia'] > rec_med and row['Frequencia'] >= freq_med:
        return 'Em Risco'
    else:
        return 'Esporádicos'

# Aplicar nomeação ao perfil
perfil['Nome_Segmento'] = perfil.apply(nomear_cluster, axis=1)

print('Perfil com Nomes dos Segmentos:')
print('=' * 80)
print(perfil[['Nome_Segmento', 'N_Clientes', 'Pct_Clientes', 'Recencia', 'Frequencia', 'Monetario']])

# Salvar perfil
perfil.to_csv('../reports/perfil_clusters_rfm.csv')
print('\nPerfil salvo: reports/perfil_clusters_rfm.csv')

In [ ]:
# =============================================================================
# Célula 16: Visualização dos Clusters (Scatter + Bar)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scatter: Recência vs Monetário
colors_map = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71', 3: '#f39c12'}
for cluster_id in range(K_FINAL):
    mask = rfm['Cluster'] == cluster_id
    nome = perfil.loc[cluster_id, 'Nome_Segmento'] if 'Nome_Segmento' in perfil.columns else f'Cluster {cluster_id}'
    axes[0].scatter(
        rfm.loc[mask, 'Recencia'],
        rfm.loc[mask, 'Monetario'],
        alpha=0.3, label=nome, color=colors_map[cluster_id], s=15
    )
axes[0].set_title('Segmentos: Recência vs Monetário', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Recência (dias)')
axes[0].set_ylabel('Monetário (R$)')
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, rfm['Monetario'].quantile(0.95))  # Limitar outliers visuais
axes[0].grid(True, alpha=0.3)

# Bar chart: Tamanho dos clusters
cluster_sizes = rfm['Cluster'].value_counts().sort_index()
nomes = [perfil.loc[i, 'Nome_Segmento'] if 'Nome_Segmento' in perfil.columns else f'C{i}' for i in cluster_sizes.index]
bars = axes[1].bar(nomes, cluster_sizes.values, 
                   color=[colors_map[i] for i in cluster_sizes.index],
                   edgecolor='white', linewidth=1.5)
axes[1].set_title('Distribuição de Clientes por Segmento', fontsize=13, fontweight='bold')
axes[1].set_ylabel('N. Clientes')
for bar, val in zip(bars, cluster_sizes.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}', ha='center', fontweight='bold')

plt.suptitle('Segmentação de Clientes (RFM + K-Means)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/modelo_6_clusters_rfm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: reports/figures/modelo_6_clusters_rfm.png')

In [ ]:
# =============================================================================
# Célula 17: Radar Chart dos Clusters (Perfil Normalizado)
# =============================================================================

# Normalizar perfil para radar (0-1)
perfil_norm = perfil[['Recencia', 'Frequencia', 'Monetario']].copy()
# Inverter recência (menor = melhor)
r_range = perfil_norm['Recencia'].max() - perfil_norm['Recencia'].min()
perfil_norm['Recencia'] = 1 - (perfil_norm['Recencia'] - perfil_norm['Recencia'].min()) / (r_range if r_range > 0 else 1)
f_range = perfil_norm['Frequencia'].max() - perfil_norm['Frequencia'].min()
perfil_norm['Frequencia'] = (perfil_norm['Frequencia'] - perfil_norm['Frequencia'].min()) / (f_range if f_range > 0 else 1)
m_range = perfil_norm['Monetario'].max() - perfil_norm['Monetario'].min()
perfil_norm['Monetario'] = (perfil_norm['Monetario'] - perfil_norm['Monetario'].min()) / (m_range if m_range > 0 else 1)

categories = ['Recencia\n(Recente)', 'Frequencia', 'Monetario']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Fechar o polígono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for cluster_id in range(K_FINAL):
    values = perfil_norm.loc[cluster_id].values.flatten().tolist()
    values += values[:1]
    nome = perfil.loc[cluster_id, 'Nome_Segmento'] if 'Nome_Segmento' in perfil.columns else f'Cluster {cluster_id}'
    ax.plot(angles, values, 'o-', linewidth=2, label=nome, color=colors_map[cluster_id])
    ax.fill(angles, values, alpha=0.1, color=colors_map[cluster_id])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_title('Perfil dos Segmentos (Radar RFM)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/modelo_7_radar_rfm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: reports/figures/modelo_7_radar_rfm.png')

In [ ]:
# =============================================================================
# Célula 18: Heatmap do Perfil RFM por Cluster
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Usar perfil para heatmap
perfil_display = perfil[['Recencia', 'Frequencia', 'Monetario']].copy()
nomes_idx = [perfil.loc[i, 'Nome_Segmento'] if 'Nome_Segmento' in perfil.columns else f'C{i}' for i in perfil.index]
perfil_display.index = nomes_idx

sns.heatmap(perfil_display, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=2, linecolor='white')
ax.set_title('Perfil Médio dos Segmentos RFM', fontsize=14, fontweight='bold')
ax.set_ylabel('Segmento')

plt.tight_layout()
plt.savefig('../reports/figures/modelo_8_heatmap_perfil.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva: reports/figures/modelo_8_heatmap_perfil.png')

---
## Resumo dos Resultados

### Previsão de Satisfação
- **Logistic Regression** e **Random Forest** foram treinados para classificar pedidos como satisfeitos/insatisfeitos
- As features mais importantes são: **delivery_delay**, **delivery_days**, **freight_value** e **price**
- O cumprimento do prazo de entrega é o fator mais crítico para a satisfação do cliente

### Segmentação RFM
- Foram identificados **4 segmentos** de clientes via K-Means
- **VIP**: Compras recentes + alto valor monetário
- **Novos/Promissores**: Compras recentes + menor valor (potencial de crescimento)
- **Em Risco**: Alta frequência passada mas sem compras recentes
- **Esporádicos**: Baixa frequência e baixo valor

### Ficheiros Gerados
- `reports/figures/modelo_1_curva_roc.png` — Curvas ROC comparativas
- `reports/figures/modelo_2_confusion_matrix.png` — Matrizes de confusão
- `reports/figures/modelo_3_feature_importance.png` — Importância das features
- `reports/figures/modelo_4_rfm_distribuicao.png` — Distribuição RFM
- `reports/figures/modelo_5_elbow_silhouette.png` — Elbow + Silhouette
- `reports/figures/modelo_6_clusters_rfm.png` — Visualização dos clusters
- `reports/figures/modelo_7_radar_rfm.png` — Radar chart dos segmentos
- `reports/figures/modelo_8_heatmap_perfil.png` — Heatmap do perfil
- `reports/tabela_comparativa_modelos.csv` — Tabela de métricas
- `reports/perfil_clusters_rfm.csv` — Perfil dos clusters